# Exploratory Data Analysis

This notebook explores the raw Titanic dataset to understand the structure, quality, and modeling readiness of the data before feature testing and model development.

The analysis begins with basic data inspection, including dataset shape, duplicate checks, data types, missing values, and summary statistics. Missing values are handled based on the context of each feature: cabin information is preserved as an `"Unknown"` category, age is imputed using the average passenger age, and embarked values are filled using the most frequent category.

The notebook then reviews relationships between variables using correlation analysis and visualizations. Numeric distributions for `Age` and `Fare` are examined to identify skewness and potential outliers. Because `Fare` contains high-leverage values, a log transformation is applied to reduce skewness and make the feature more stable for downstream modeling.

Categorical variables are then prepared for modeling. `Sex` and `Embarked` are converted into dummy variables, while `Cabin` is target encoded because it is a high-cardinality categorical feature. Numeric variables such as `Age`, `Fare`, `Pclass`, and `SibSp` are standardized so they can be used effectively in models sensitive to feature scale.

The final section creates engineered interaction and ratio features to capture relationships between passenger demographics, fare, family size, and class. The completed dataset is saved as `uncut_feat.csv` for downstream feature testing, hypothesis testing, and model development.

In [ ]:
# Load libraries
import pandas as pd # data manipulation
from pathlib import Path # Locate script directory
import numpy as np # vectorized math


In [ ]:
# load dataset 

# get the directory where the current script is saved
dir = Path.cwd()

# point to data file (go up into workflow and down into data)
data_path = dir.parent / 'data' / 'raw_data.csv'

# load csv
df1 = pd.read_csv(data_path)

In [ ]:
# view shape info
df1.shape

In [ ]:
# duplicate info count 
df1.duplicated().sum()

In [ ]:
# review data types (helps avoids data mining issue later downstream)
df1.dtypes

In [ ]:
# check for null values 
df1.isnull().sum()

In [ ]:
# remove null values (In this cell, instead of removing nan values we will turn the column into a signal itself -> "Unkwnown")

df1['Cabin'] = df1['Cabin'].fillna('Unknown')

In [ ]:
# replace null values with average age (mean)

mean = df1['Age'].mean().astype(int)

df1['Age'] = df1['Age'].fillna(mean)

In [1]:
 # replace null values with most frequent qualatative value (mode)

mode = df1['Embarked'].mode()[0]

df1['Embarked'] = df1['Embarked'].fillna(mode)

NameError: name 'df1' is not defined

In [ ]:
# check results of null removal

df1.isnull().sum()

In [ ]:
# remove "Name" column 
df1 = df1.drop(columns = ['Name'])

In [ ]:
# summary statistics 
df1.describe(include=['number']).T

In [ ]:
#                                   --- Visualization Phase--- 
import seaborn as sns 
import matplotlib.pyplot as plt

In [ ]:
# Create a correlation heatmap

# drop the ID column... this variable has no use in training, will effect the visualization 
df_drop = df1.drop(columns= ['PassengerId']) 

# Create boolean columns for categorical features 
df_dummy = pd.get_dummies(df1[['Embarked', 'Sex']])

# merge boolean dataframe with original
df1 = df_drop.merge(df_dummy, left_index= True, right_index= True)

# compute correlation matrix for numeric values only 
corr = df1.select_dtypes(include = ['number', 'bool']).corr() # grab all numbers and boolean columns and store into corr object

# create map
plt.figure(figsize= (6,7)) # visualization size 
sns.heatmap(corr, annot = False, cmap= 'coolwarm', fmt = '.2f' ) # correation matrix plot 

In [ ]:
df1.shape

In [ ]:
sns.heatmap(corr, annot = False, cmap= 'coolwarm', fmt = '.2f' ) # correation matrix plot 

In [ ]:
# numeric correlation
df1.corr(numeric_only= True)

In [ ]:
# distrbution of age 
plt.figure(figsize=(6,7))
plt.hist(x= df1['Age'], density= False, )
plt.title('Distribution of Age')
plt.show() 

In [ ]:
# Distribution of Age with smooth =
sns.kdeplot(data = df1, x = 'Age', color= 'red')
plt.title('Distribution of Age')
plt.show()

In [ ]:
# Distribution of Fare (Passenger Price)
plt.figure(figsize=(6,7)) 
plt.hist(x = df1['Fare'], density= False)
plt.title('Distribution of Fare')
plt.title

In [ ]:
# Distribution of Fare with smoothing 
sns.kdeplot(df1, x = 'Fare', color = 'red')

In [ ]:
df1

In [ ]:
# Look for high leverage data plots within input variables 

# whisker plot 
plt.boxplot(df1['Age'], vert = False)
plt.title('Age')
plt.show()

In [ ]:
# Look for high leverage values in X 

# boxplot 
plt.boxplot(df1['Fare'], vert = False)
plt.title("Fare")
plt.show()

In [ ]:
# The Fare varibale has data points far from the residual 
# To avoid the regression line being pull towards them and distorting the decision boudary, I will log transform the columns indiviually

# Fare log transformation 
df1['Fare'] = np.log1p(df1['Fare'])


In [ ]:
# View distribution for Fare variable 
plt.figure(figsize= (6,5)) 
sns.kdeplot(data = df1, x = 'Fare', color= 'red')
plt.title('Distribution  of Far (Log Transformed')
plt.show()

In [ ]:
# Let's view the plot using a Q-Q plot 

import scipy.stats as stats
stats.probplot(df1['Fare'], dist = 'norm', plot = plt)
plt.title('Q-Q plot of Fare')
plt.show()

In [ ]:
# turn high cardinality varibles into meaningful patterns
# Model can learn from the categories based on the mean of each category coupled with the target variable 

from sklearn.preprocessing import TargetEncoder # library 

# intialize encoder (including cross validation prohibits data leakage)
encoder = TargetEncoder(smooth = 'auto', cv = 5, shuffle = True, random_state= 42)

# store variables into X and Y obejects
X = df1[['Cabin']]
Y = df1['Survived']

# fit to transform 
df1['Cabin_encoded'] = encoder.fit_transform(X, Y)

In [ ]:
# scale numerical values (This is leads to model converging much faster and avoids large values dominating the model downstream)
from sklearn.preprocessing import StandardScaler

# create instance
scaler = StandardScaler()

# store columns in a object
X = df1[['Age']]
X1 = df1[['Fare']]
X2 = df1[['Pclass']]
X3 = df1[['SibSp']]

# fit to transform columns
df1['Age'] = scaler.fit_transform(X)
df1['Fare'] = scaler.fit_transform(X1)
df1['Pclass'] = scaler.fit_transform(X2)
df1['SibSp'] = scaler.fit_transform(X3) 

In [ ]:
df1

In [ ]:
# remove ticket number column (Weak or meanless variable)
df1 = df1.drop(columns=['Ticket'])

In [ ]:
# remove uneccesary columns (Stings won't fit in training downstream)
df1 = df1.drop(columns=['Sex'])
df1 = df1.drop(columns=['Cabin'])
df1 = df1.drop(columns=["Embarked"])

In [ ]:
df1

In [ ]:
# Feature Engineneering Phase 
# (Later we will test t-test coefficients to view if feature is statistically significant in predicting target)

# Interaction features  
df1['interaction1'] = (df1['SibSp'] * df1['Parch'])
df1['interaction2'] = (df1['SibSp'] * df1['Fare'])
df1['interaction3'] = (df1['Parch'] * df1['Fare'])
df1['interaction4'] = (df1['Age'] * df1['Fare'])
df1['interaction5'] = (df1['Pclass'] * df1['Fare'])
df1['interaction6'] = (df1['Sex_female'] * df1['Fare'])
df1['interaction7'] = (df1['Sex_male'] * df1['Fare'])


# Ratio Features 
df1['Ratio1'] = (df1['Age'] / df1['Fare'])
df1['Ratio2'] = (df1['SibSp'] / df1['Fare'])
df1['Ratio3'] = (df1['Parch'] / df1['Fare'])
df1['Ratio4'] = (df1['Sex_female'] / df1['Fare'])
df1['Ratio5'] = (df1['Sex_male'] / df1['Fare'])





In [ ]:
# save the dataset with engineered features to Data dir as a csv

save_path = dir.parent / 'Data' / 'uncut_feat.csv'

# save to directed path
df1.to_csv(save_path, index=False)
